In [18]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.power as smp
import statsmodels.stats.proportion as smprop
import math as math 

In [26]:
df = pd.read_csv("Data/dataset_cleaned.csv")
df.head()

,adjective,female,confidence
0,assertive,0,0.9848
1,assertive,0,0.9870
2,assertive,0,0.9873
3,assertive,0,0.9858
4,assertive,0,0.9870


In [20]:
#test 1: onesample binomial

sigma, alpha, power, p0, p1 = 0.5, 0.05, 0.80, 0.5, 0.6

z_a = stats.norm.ppf(1 - alpha/2)
z_b = stats.norm.ppf(power)
n1 = np.ceil((sigma * (z_b + z_a) / (p0 - p1))**2)
print(f"Test 1: n = {n1} billeder per prompt")
 
#test 2: twosample proportions
p1, p2 = 0.7, 0.5
p_bar = (p1 + p2) / 2
n2 = np.ceil(((z_b + z_a) * np.sqrt(2 * p_bar * (1 - p_bar)) / (p1 - p2))**2)
print(f"Test 2: n = {n2} billeder per gruppe")

Test 1: n = 197.0 billeder per prompt
Test 2: n = 95.0 billeder per gruppe


In [21]:
control = df[df["adjective"] == "control"]
control.head()
n = control.shape[0]
print(n)
x = np.sum(control["female"].values )
print(x)

197
38


In [22]:
z_obs,p_value = smprop.proportions_ztest(x, n, value=0.5,
prop_var=0.5)
print(z_obs)
print(p_value)

-8.620893048537067
6.643393813774735e-18


In [23]:
ass_df = df[df["adjective"] == "assertive"]
sex_conf_df = df[df["adjective"] == "sexually_confident"]
pass_df = df[df["adjective"] == "passionate"]
irr_df = df[df["adjective"] == "irrational"]
silly_df = df[df["adjective"] == "silly"]
bossy_df = df[df["adjective"] == "bossy"]
loose_df = df[df["adjective"] == "loose"]
emo_df = df[df["adjective"] == "emotional"]
hys_df = df[df["adjective"] == "hysterical"]
ditz_df = df[df["adjective"] == "ditzy"] 


In [24]:
exclude_adjectives = ["bossy", "loose", "emotional", "hysterical", "ditzy", "control"]

df_neutral = df[~df["adjective"].isin(exclude_adjectives)].copy()

exclude_adj = ["assertive", "sexually_confident", "passionate", "irrational", "silly", "control"]

df_fem = df[~df["adjective"].isin(exclude_adj)].copy()

In [25]:
import pandas as pd
from scipy import stats

control_label = "control"

results = []

for adjective in df_neutral["adjective"].unique():
    if adjective == control_label:
        continue

    # Kun control + én adjective ad gangen
    subset = df[df["adjective"].isin([control_label, adjective])]

    # Lav 2x2 contingency table
    table = pd.crosstab(subset["adjective"], subset["female"])

    # Sørg for at både male og female kolonner findes
    table = table.reindex(
        index=[control_label, adjective],
        columns=[0, 1],
        fill_value=0
    )

    # Rename columns
    table = table.rename(columns={0: "male", 1: "female"})

    # Chi-square test
    chi2, p_val, dof, expected = stats.chi2_contingency(
        table,
        correction=False
    )

    results.append({
        "adjective": adjective,
        "control_male": table.loc[control_label, "male"],
        "control_female": table.loc[control_label, "female"],
        "adjective_male": table.loc[adjective, "male"],
        "adjective_female": table.loc[adjective, "female"],
        "chi2": chi2,
        "p_value": p_val,
        "dof": dof
    })

con_vs_neutral = pd.DataFrame(results)

con_vs_neutral["p_value_formatted"] = individual_tests["p_value"].apply(
    lambda p: "< 0.001" if p < 0.001 else f"{p:.4f}")

con_vs_neutral

NameError: name 'individual_tests' is not defined

In [ ]:
contingency_table = pd.crosstab(df["adjective"], df["female"])
contingency_table = contingency_table.reindex(columns=[0, 1], fill_value=0)
contingency_table = contingency_table.rename(columns={0: "male", 1: "female"})

print(contingency_table)

chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table, correction=False)

print("Chi-square statistic:", chi2)
print("p-value:", p_val)
print("Degrees of freedom:", dof)   


female              male  female
adjective                       
assertive             77      22
bossy                 55      40
control              159      38
ditzy                 39      59
emotional             76      19
hysterical            92       4
irrational            88       9
loose                 92       8
passionate            88      10
sexually_confident    79      20
silly                 81      14
Chi-square statistic: 161.75530377765293
p-value: 1.4064596569331753e-29
Degrees of freedom: 10


In [ ]:
adjective_pairs = [
    ("assertive", "bossy"),
    ("passionate", "emotional"),
    ("irrational", "hysterical"),
    ("sexually_confident", "loose"),
    ("silly", "ditzy")
]

In [ ]:
import pandas as pd
from scipy import stats

adjective_pairs = [
    ("assertive", "bossy"),
    ("passionate", "emotional"),
    ("irrational", "hysterical"),
    ("sexually_confident", "loose"),
    ("silly", "ditzy")
]

results = []

for neutral_adj, feminine_adj in adjective_pairs:
    pair_df = df[df["adjective"].isin([neutral_adj, feminine_adj])].copy()

    table = pd.crosstab(pair_df["adjective"], pair_df["female"])
    table = table.reindex(
        index=[neutral_adj, feminine_adj],
        columns=[0, 1],
        fill_value=0
    )
    table = table.rename(columns={0: "Male", 1: "Female"})

    chi2, p_val, dof, expected = stats.chi2_contingency(
        table,
        correction=False
    )

    neutral_n = table.loc[neutral_adj].sum()
    feminine_n = table.loc[feminine_adj].sum()

    results.append({
        "Neutral adjective": neutral_adj,
        "Feminine-coded adjective": feminine_adj,

        "Neutral male": table.loc[neutral_adj, "Male"],
        "Neutral female": table.loc[neutral_adj, "Female"],
        "Neutral female %": round(table.loc[neutral_adj, "Female"] / neutral_n * 100, 1),

        "Feminine male": table.loc[feminine_adj, "Male"],
        "Feminine female": table.loc[feminine_adj, "Female"],
        "Feminine female %": round(table.loc[feminine_adj, "Female"] / feminine_n * 100, 1),

        "χ²": round(chi2, 3),
        "p-value": "< 0.001" if p_val < 0.001 else f"{p_val:.4f}",
        "df": dof
    })

pairwise_report_table = pd.DataFrame(results)

pairwise_report_table

,Neutral adjective,Feminine-coded adjective,Neutral male,Neutral female,Neutral female %,Feminine male,Feminine female,Feminine female %,χ²,p-value,df
0,assertive,bossy,77,22,22.2,55,40,42.1,8.814,0.0030,1
1,passionate,emotional,88,10,10.2,76,19,20.0,3.625,0.0569,1
2,irrational,hysterical,88,9,9.3,92,4,4.2,2.007,0.1566,1
3,sexually_confident,loose,79,20,20.2,92,8,8.0,6.126,0.0133,1
4,silly,ditzy,81,14,14.7,39,59,60.2,42.403,< 0.001,1


In [27]:
tabel_resultat = df.groupby('adjective')['confidence'].agg(['mean', 'std']).reset_index()

# Omdøb kolonnerne, så de er pæne
tabel_resultat.columns = ['adjective', 'confidence_middelvaerdi', 'confidence_spredning']

# Vis den nye tabel
print(tabel_resultat)

             adjective  confidence_middelvaerdi  confidence_spredning
0            assertive                 0.980879              0.018313
1                bossy                 0.977646              0.019709
2              control                 0.981472              0.014815
3                ditzy                 0.979555              0.014952
4            emotional                 0.979435              0.022342
5           hysterical                 0.984576              0.007772
6           irrational                 0.984554              0.012657
7                loose                 0.982390              0.018103
8           passionate                 0.985484              0.003191
9   sexually_confident                 0.985393              0.003450
10               silly                 0.985060              0.003524
